In [ ]:
pip install gensim nltk

In [ ]:
import re
import pandas as pd
import numpy as np
import nltk
from nltk.tokenize import sent_tokenize,word_tokenize
from gensim.models import Word2Vec
nltk.download('stopwords')
from nltk.corpus import stopwords

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
# Download required NLTK data (run once)
nltk.download('punkt')
# Note: In newer NLTK versions you might need 'punkt_tab'
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [ ]:
with open('A Game Of Thrones.txt','r',encoding='utf-8') as file:
  text=file.read()

In [ ]:
with open('Harry_Potter_Book_Text.txt','r',encoding='utf-8') as file:
  potter_data_1=file.read()

In [ ]:
print(text[:1000])

A Game Of Thrones 
Book One of A Song of Ice and Fire 
By George R. R. Martin 
PROLOGUE 
"We should start back," Gared urged as the woods began to grow dark around them. "The wildlings are 
dead." 
"Do the dead frighten you?" Ser Waymar Royce asked with just the hint of a smile. 
Gared did not rise to the bait. He was an old man, past fifty, and he had seen the lordlings come and go. 
"Dead is dead," he said. "We have no business with the dead." 
"Are they dead?" Royce asked softly. "What proof have we?" 
"Will saw them," Gared said. "If he says they are dead, that's proof enough for me." 
Will had known they would drag him into the quarrel sooner or later. He wished it had been later rather 
than sooner. "My mother told me that dead men sing no songs," he put in. 
"My wet nurse said the same thing, Will," Royce replied. "Never believe anything you hear at a woman's 
tit. There are things to be learned even from the dead." His voice echoed, too loud in the twilit forest. 
Page 1

"We h

In [ ]:
stop_words = set(stopwords.words('english'))
negation_words = {'not', 'no', 'neither', 'nor'}
stop_words = stop_words - negation_words

In [ ]:
def preprocess_word2vec(text):

    # 1. Sentence tokenization
    sentences = sent_tokenize(text)

    cleaned_sentences = []

    for sentence in sentences:

        # 2. Lowercase
        sentence = sentence.lower()

        # 3. Word tokenization FIRST
        words = word_tokenize(sentence)

        # 4. Keep only alphabetic words
        words = [word for word in words if word.isalpha()]

        # 5. Remove stopwords
        words = [word for word in words if word not in stop_words]

        # 6. Keep non-empty sentences
        if words:
            cleaned_sentences.append(words)

    return cleaned_sentences

In [ ]:
# Game of Thrones data
cleaned_data=preprocess_word2vec(text)

In [ ]:
# Harry potter data-1
cleaned_data_potter_1=preprocess_word2vec(potter_data_1)

In [ ]:
# Train the Word2Vec model - Game of Thrones
model = Word2Vec(
    sentences=cleaned_data,
    vector_size=100,  # Size of the dense vector to represent each word
    window=5,         # Maximum distance between current and predicted word
    min_count=2,      # Ignores all words with total frequency lower than this
    sg=1,
    epochs=50,        # Number of training epochs
    workers=4         # Number of CPU threads to use for training
)

In [ ]:
# Train the Word2Vec model  - Harry Potter
model = Word2Vec(
    sentences=cleaned_data_potter_1,
    vector_size=100,  # Size of the dense vector to represent each word
    window=5,         # Maximum distance between current and predicted word
    min_count=2,      # Ignores all words with total frequency lower than this
    sg=1,
    epochs=50,        # Number of training epochs
    workers=4         # Number of CPU threads to use for training
)

In [ ]:
# 1. Who is most associated with Voldemort?
print(model.wv.most_similar('voldemort', topn=5))
# Expected output: 'prophecy', 'wormtail', 'bellatrix', 'power', 'master'

# 2. Who is most associated with the Malfoy family?
print(model.wv.most_similar('malfoy', topn=3))
# Expected output: 'goyle', 'crabbe', 'narcissa'

# 3. Semantic clustering around verbs
print(model.wv.most_similar('fight', topn=4))
# Expected output: 'kill', 'die', 'control', 'avada_kedavra'

# 4. Which entity does not belong?
print(model.wv.doesnt_match(['gryffindor', 'slytherin', 'hufflepuff', 'azkaban']))
# Expected output: 'azkaban' (The model will cluster the first three as Hogwarts houses, while Azkaban is a prison)

# 5. Character Analogies: Harry is to Gryffindor as Draco is to...?
# harry_potter - gryffindor + slytherin = ?
print(model.wv.most_similar(positive=['harry', 'potter', 'slytherin'], negative=['gryffindor'], topn=5))
# Expected output: 'draco_malfoy'

# 6. Relational Analogies: Harry is to Ron as Draco is to...?
# harry_potter - ron_weasley + crabbe = ?
print(model.wv.most_similar(positive=['harry', 'potter', 'crabbe'], negative=['ron', 'weasley'], topn=5))
# Expected output: 'draco_malfoy'

[('lord', 0.7455926537513733), ('debt', 0.583997368812561), ('ambitions', 0.5716915130615234), ('thrice', 0.5642257332801819), ('torturing', 0.5413469672203064)]
[('draco', 0.7414807677268982), ('lucius', 0.6973815560340881), ('crabbe', 0.6511055827140808)]
[('cur', 0.5884110331535339), ('tribes', 0.535882294178009), ('defender', 0.5316751003265381), ('courageous', 0.5294721126556396)]
azkaban
[('z', 0.8433911204338074), ('rowling', 0.8356544971466064), ('page', 0.8101726770401001), ('sn', 0.6838758587837219), ('thieving', 0.6800175309181213)]
[('page', 0.6565321087837219), ('rowling', 0.6391112804412842), ('z', 0.6172254681587219), ('whiplike', 0.5683116912841797), ('unheard', 0.5522329211235046)]


In [ ]:
# 1. Who is associated with Daenerys?
print(f" Daenerys associated:{model.wv.most_similar('daenerys', topn=5)}")
# Expected output: 'targaryen', 'dragons', 'jorah', 'khal_drogo'

print("=================================================")
# 2. Which house doesn't belong?
print(model.wv.doesnt_match(['stark', 'lannister', 'baratheon', 'wildling']))
# Expected output: 'wildling' (the others are noble houses)


print("=================================================")
# 3. Family analogies: Jon is to Stark as Tyrion is to...?
# Jon - Stark + Lannister = ?
print(model.wv.most_similar(positive=['jon', 'snow', 'lannister'], negative=['stark'], topn=1))
# Expected output: 'tyrion' or 'jaime'


print("=================================================")
# 4. Explore locations
print(model.wv.most_similar('winterfell'))

 Daenerys associated:[('stormborn', 0.7159505486488342), ('targaryen', 0.6694769859313965), ('unburnt', 0.6255438923835754), ('quicken', 0.6173532605171204), ('arouse', 0.5873547792434692)]
wildling
[('pyp', 0.5521856546401978)]
[('stark', 0.643123984336853), ('eddard', 0.6230976581573486), ('robb', 0.5890122056007385), ('rickon', 0.5831974148750305), ('dreadfort', 0.5788120031356812), ('berena', 0.561927080154419), ('riverrun', 0.5538513660430908), ('motherless', 0.5465866923332214), ('granite', 0.5401402711868286), ('harrenhal', 0.5389182567596436)]
